In [1]:
import math
import pandas as pd
from ace_tools import display_dataframe_to_user

# --- Physical and model parameters ---
# UV geometric scale (MeV)
B_UV = 140.45

# QCD parameters for RG running
alpha_s_MZ = 0.1181        # alpha_s at MZ
MZ = 91.1876               # Z mass in GeV
mp_GeV = 0.9382720813      # proton mass in GeV
Nf = 5                     # number of active flavors
b0 = 11 - 2/3 * Nf         # one-loop beta function coefficient

# Skyrme model parameters
F_pi = 186.0               # effective pion decay constant in MeV
e_SK = 4.84                # Skyrme parameter
f_profile = 0.25           # profile functional factor

# Unit conversions
hbarc = 197.3269804        # MeV·fm
fm_to_invMeV = 1 / hbarc   # fm -> MeV^-1

# --- 1. RG dressing of B ---
rho = 1 + (b0 * alpha_s_MZ / (2 * math.pi)) * math.log(MZ / mp_GeV)
B_IR = B_UV * rho  # MeV

# --- 2. Hopf-tower volumes V_n ---
V = {n: math.pi**n / math.factorial(n) for n in [2, 3, 4, 5]}

# --- 3. Solve universal self-energies Delta_ud, Delta_s ---
# Experimental masses (MeV)
m_p = 938.2720813
m_Sigma = (1189 + 1193 + 1385) / 3
m_Xi = (1314.9 + 1321.7 + 1531.8) / 3

# Solve: B_IR*V5 + 3*Delta_ud = m_p  => Delta_ud
Delta_ud = (m_p - B_IR * V[5]) / 3

# Solve: B_IR*V4 + 2*Delta_ud + Delta_s = m_Sigma  => Delta_s
Delta_s = m_Sigma - (B_IR * V[4] + 2 * Delta_ud)

# --- 4. Derive spin gap Delta_spin via Skyrmion moment of inertia ---
# 4.1 Static Skyrmion energy identified with B_IR * V5
E_cl = B_IR * V[5]  # MeV

# 4.2 Effective Skyrmion radius (in fm)
R = ((2 * E_cl) / (math.pi**2 * F_pi**2))**(1/3)

# 4.3 Moment of inertia (in fm)
I_fm = (8 * math.pi / 3) * F_pi**2 * R**3 * f_profile * (1 / hbarc**2)

# Convert to MeV^-1
I_inv = I_fm * (1 / fm_to_invMeV)

# 4.4 Spin-excitation gap
Delta_spin = (3 / 2) * (1 / I_inv)  # MeV

# --- 5. Predict baryon masses ---
baryons = [
    ("p",       5, 3, 0, 0, m_p),
    ("n",       5, 3, 0, 0, 939.5654133),
    ("Sigma",   4, 2, 1, 0, m_Sigma),
    ("Xi",      3, 1, 2, 0, m_Xi),
    ("Omega",   2, 0, 3, 1, 1672.45),
    ("Delta",   5, 3, 0, 1, 1232.0),
    ("Sigma*",  4, 2, 1, 1, 1385.0),
    ("Xi*",     3, 1, 2, 1, 1531.8),
]

pred_rows = []
for name, n, Nud, Ns, S, m_exp in baryons:
    m_pred = B_IR * V[n] + Nud * Delta_ud + Ns * Delta_s + S * Delta_spin
    pred_rows.append({
        "Baryon": name,
        "n": n,
        "N_ud": Nud,
        "N_s": Ns,
        "Spin S": S,
        "Predicted (MeV)": round(m_pred, 2),
        "Experimental (MeV)": round(m_exp, 2),
        "Ratio Pred/Exp": round(m_pred / m_exp, 3)
    })

# Display results
df_params = pd.DataFrame([{
    "B_IR (MeV)": round(B_IR, 2),
    "Delta_ud (MeV)": round(Delta_ud, 2),
    "Delta_s (MeV)": round(Delta_s, 2),
    "Delta_spin (MeV)": round(Delta_spin, 2),
    "Skyrmion radius R (fm)": round(R, 2),
    "Moment of inertia I (MeV^-1)": round(I_inv, 2)
}])

display_dataframe_to_user(name="Model Parameters After RG & Skyrme", dataframe=df_params)
display_dataframe_to_user(name="Baryon Mass Predictions", dataframe=pd.DataFrame(pred_rows))

ModuleNotFoundError: No module named 'ace_tools'